# Regression


In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

# 1. Data
df = pd.read_csv("creditsense-ai1215/credit_train.csv")  # Load your dataset

# PreProcessing

X  = df.drop(['RiskTier', 'InterestRate'], axis=1)
y_reg = df['InterestRate']


In [7]:
def simple_preprocess(X_train, X_test=None):
    """
    Fill missing values, add missing-value indicators, and encode categorical features.
    """
    X_train = X_train.copy()
    if X_test is not None:
        X_test = X_test.copy()

    base_cols = X_train.columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    # Add missing indicators BEFORE imputation so the model can learn missingness effects.
    missing_indicator_cols = []
    for col in base_cols:
        train_missing = X_train[col].isna()
        test_missing = X_test[col].isna() if X_test is not None else None

        if train_missing.any() or (test_missing is not None and test_missing.any()):
            ind_col = f"{col}_is_missing"
            X_train[ind_col] = train_missing.astype(np.int8)
            if X_test is not None:
                X_test[ind_col] = test_missing.astype(np.int8)
            missing_indicator_cols.append(ind_col)

    print(
        f"Numeric: {len(num_cols)} | Categorical: {len(cat_cols)} | "
        f"Missing indicators: {len(missing_indicator_cols)}"
    )

    # Numeric -> fill with train median
    for col in num_cols:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        if X_test is not None:
            X_test[col] = X_test[col].fillna(med)

    # Categorical -> fill with train mode, then label encode
    for col in cat_cols:
        mode_series = X_train[col].mode(dropna=True)
        mode = mode_series.iloc[0] if not mode_series.empty else 'Unknown'

        X_train[col] = X_train[col].fillna(mode)
        if X_test is not None:
            X_test[col] = X_test[col].fillna(mode)

        le = LabelEncoder()
        if X_test is not None:
            combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)], axis=0)
            le.fit(combined)
            X_train[col] = le.transform(X_train[col].astype(str))
            X_test[col] = le.transform(X_test[col].astype(str))
        else:
            X_train[col] = le.fit_transform(X_train[col].astype(str))

    X_train = X_train.fillna(0)
    if X_test is not None:
        X_test = X_test.fillna(0)
        return X_train, X_test

    return X_train


X_processed = simple_preprocess(X)
print(f"\nAfter preprocessing — shape: {X_processed.shape} | missing: {X_processed.isnull().sum().sum()}")


Numeric: 46 | Categorical: 9 | Missing indicators: 16

After preprocessing — shape: (35000, 71) | missing: 0


In [9]:
# 2. Split - use preprocessed data
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_reg, test_size=0.2, random_state=42
)

# 3. Model
model_1 = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

# 4. Train
model_1.fit(X_train, y_train)

# 5. Predict
y_pred = model_1.predict(X_test)

# 6. Evaluate
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

MSE: 3.182456725228973


In [11]:
# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("creditsense-ai1215/credit_train.csv")  # Load your dataset

# Replace with your target column

X  = df.drop(['RiskTier', 'InterestRate'], axis=1)
y_reg = df['InterestRate']

# =========================
# 2. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)


# =========================
# 3. COLUMN TYPES
# =========================
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()


# =========================
# 4. PREPROCESSING
# =========================

# For tree models / linear model:
# - numeric: median imputation
# - categorical: most frequent + one-hot
basic_numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

basic_categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

basic_preprocessor = ColumnTransformer([
    ("num", basic_numeric_preprocessor, numeric_features),
    ("cat", basic_categorical_preprocessor, categorical_features)
])

# For neural network:
# same as above, but numeric features are scaled
nn_numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

nn_categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

nn_preprocessor = ColumnTransformer([
    ("num", nn_numeric_preprocessor, numeric_features),
    ("cat", nn_categorical_preprocessor, categorical_features)
])



In [12]:

# =========================
# 5. BASE MODELS
# =========================

xgb_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ))
])

rf_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

hgb_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    ))
])

lr_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", LinearRegression())
])

mlp_model = Pipeline([
    ("preprocess", nn_preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size="auto",
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=42
    ))
])


# =========================
# 6. STACKING MODEL
# =========================
base_models = [
    ("xgb", xgb_model),
    ("rf", rf_model),
    ("hgb", hgb_model),
    ("lr", lr_model),
    ("mlp", mlp_model),
]

stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
    passthrough=False
)


# =========================
# 7. TRAIN
# =========================
stack_model.fit(X_train, y_train)


# =========================
# 8. PREDICT
# =========================
y_pred = stack_model.predict(X_test)


# =========================
# 9. EVALUATE
# =========================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Stacked Model Results")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")

Stacked Model Results
MSE  : 2.9077
RMSE : 1.7052
MAE  : 1.3433
R2   : 0.8248


In [16]:
test_df = pd.read_csv("creditsense-ai1215/credit_test.csv")

X_test_final = test_df
y_test_final_pred = stack_model.predict(X_test_final)

test_submission = pd.DataFrame({
    "InterestRate": y_test_final_pred
})
test_submission.to_csv("gukas_reg_submission.csv")